# 22. Robotics action policies — ACT, Diffusion Policy, π0-style flow, FAST

Model width/horizon are reduced for a T4, but each action-generation path is kept end-to-end.


In [ ]:
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. ACT — CVAE latent + learned action queries + chunk loss


In [ ]:
class TinyACT(nn.Module):
    def __init__(self, obs_dim=8, action_dim=3, chunk=4, hidden=24, latent=8):
        super().__init__()
        self.latent_dim = latent
        self.obs_proj = nn.Linear(obs_dim, hidden)
        self.action_proj = nn.Linear(action_dim, hidden)

        enc_layer = nn.TransformerEncoderLayer(hidden, 3, 4 * hidden, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.mu = nn.Linear(hidden, latent)
        self.logvar = nn.Linear(hidden, latent)

        self.latent_proj = nn.Linear(latent, hidden)
        self.queries = nn.Parameter(torch.randn(1, chunk, hidden) * 0.02)
        dec_layer = nn.TransformerDecoderLayer(hidden, 3, 4 * hidden, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=1)
        self.action_head = nn.Linear(hidden, action_dim)

    def forward(self, obs, target_chunk=None):
        batch = obs.size(0)
        obs_token = self.obs_proj(obs).unsqueeze(1)

        if target_chunk is not None:
            action_tokens = self.action_proj(target_chunk)
            encoded = self.encoder(torch.cat([obs_token, action_tokens], dim=1))[:, 0]
            mu = self.mu(encoded)
            logvar = self.logvar(encoded)
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        else:
            mu = torch.zeros(batch, self.latent_dim, device=obs.device)
            logvar = torch.zeros_like(mu)
            z = torch.zeros_like(mu)

        memory = torch.cat([obs_token, self.latent_proj(z).unsqueeze(1)], dim=1)
        queries = self.queries.expand(batch, -1, -1)
        actions = self.action_head(self.decoder(queries, memory))
        return actions, mu, logvar


obs = torch.randn(4, 8, device=device)
target_chunk = torch.randn(4, 4, 3, device=device)
act = TinyACT().to(device)
pred_chunk, mu, logvar = act(obs, target_chunk)
reconstruction = F.l1_loss(pred_chunk, target_chunk)
kl = -0.5 * torch.mean(1 + logvar - mu.square() - logvar.exp())
act_loss = reconstruction + 0.005 * kl
act_loss.backward()
print("ACT train chunk:", pred_chunk.shape)
print("ACT inference chunk:", act(obs[:1])[0].shape)


## 2. Diffusion Policy — conditional temporal 1D U-Net and iterative rollout


In [ ]:
def time_embedding(t, dim):
    half = dim // 2
    frequency = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=t.device, dtype=t.dtype)
        / max(half - 1, 1)
    )
    angle = t[:, None] * frequency[None]
    return torch.cat([angle.sin(), angle.cos()], dim=-1)


class CondRes1D(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(4, out_ch)
        self.norm2 = nn.GroupNorm(4, out_ch)
        self.condition = nn.Linear(cond_dim, 2 * out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, 1)

    def forward(self, x, condition):
        hidden = self.norm1(self.conv1(x))
        scale, shift = self.condition(F.silu(condition)).chunk(2, dim=-1)
        hidden = F.silu(hidden * (1 + scale[:, :, None]) + shift[:, :, None])
        hidden = self.norm2(self.conv2(hidden))
        return F.silu(hidden + self.skip(x))


class TinyDiffusionPolicy(nn.Module):
    def __init__(self, obs_dim=8, action_dim=3, base=16, cond_dim=32):
        super().__init__()
        self.cond_dim = cond_dim
        self.obs_cond = nn.Linear(obs_dim, cond_dim)
        self.time_mlp = nn.Sequential(nn.Linear(cond_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.in_proj = nn.Conv1d(action_dim, base, 1)
        self.down = CondRes1D(base, base, cond_dim)
        self.downsample = nn.Conv1d(base, 2 * base, 4, stride=2, padding=1)
        self.mid = CondRes1D(2 * base, 2 * base, cond_dim)
        self.upsample = nn.ConvTranspose1d(2 * base, base, 4, stride=2, padding=1)
        self.up = CondRes1D(2 * base, base, cond_dim)
        self.out = nn.Conv1d(base, action_dim, 1)

    def forward(self, noisy_actions, obs, t):
        condition = self.obs_cond(obs) + self.time_mlp(time_embedding(t, self.cond_dim))
        x = self.in_proj(noisy_actions.transpose(1, 2))
        skip = self.down(x, condition)
        x = self.mid(self.downsample(skip), condition)
        x = self.upsample(x)
        if x.size(-1) != skip.size(-1):
            x = F.interpolate(x, size=skip.size(-1), mode="linear", align_corners=False)
        x = self.up(torch.cat([x, skip], dim=1), condition)
        return self.out(x).transpose(1, 2)


@torch.no_grad()
def sample_diffusion_actions(model, obs, horizon=8, action_dim=3, steps=6):
    x = torch.randn(obs.size(0), horizon, action_dim, device=obs.device)
    for step in range(steps, 0, -1):
        t_now = torch.full((obs.size(0),), step / steps, device=obs.device)
        t_next = torch.full((obs.size(0),), (step - 1) / steps, device=obs.device)
        eps = model(x, obs, t_now)
        a_now = torch.cos(0.5 * math.pi * t_now)[:, None, None].clamp_min(1e-3)
        s_now = torch.sin(0.5 * math.pi * t_now)[:, None, None]
        x0_hat = (x - s_now * eps) / a_now
        a_next = torch.cos(0.5 * math.pi * t_next)[:, None, None]
        s_next = torch.sin(0.5 * math.pi * t_next)[:, None, None]
        x = a_next * x0_hat + s_next * eps
    return x


diffusion_policy = TinyDiffusionPolicy().to(device)
clean_actions = torch.randn(4, 8, 3, device=device)
noise = torch.randn_like(clean_actions)
t = torch.rand(4, device=device)
a = torch.cos(0.5 * math.pi * t)[:, None, None]
s = torch.sin(0.5 * math.pi * t)[:, None, None]
noisy_actions = a * clean_actions + s * noise
pred_noise = diffusion_policy(noisy_actions, obs, t)
diffusion_loss = F.mse_loss(pred_noise, noise)
diffusion_loss.backward()
sampled = sample_diffusion_actions(diffusion_policy, obs[:1])
print("Diffusion horizon:", sampled.shape, "execute prefix:", sampled[:, :2].shape)


## 3. π0-style multi-block action expert + flow rollout


In [ ]:
def pi0_mask(prefix_len, suffix_len, device):
    total = prefix_len + suffix_len
    mask = torch.zeros(total, total, dtype=torch.bool, device=device)
    mask[:prefix_len, :prefix_len] = True
    mask[prefix_len:, :] = True
    return mask


class AdaRMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.RMSNorm(dim)
        self.scale = nn.Linear(dim, dim)
        self.shift = nn.Linear(dim, dim)

    def forward(self, x, condition):
        return self.norm(x) * (1 + self.scale(condition)[:, None]) + self.shift(condition)[:, None]


class Pi0JointBlock(nn.Module):
    def __init__(self, dim=24, heads=3):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.prefix_norm = nn.RMSNorm(dim)
        self.prefix_qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.prefix_out = nn.Linear(dim, dim, bias=False)
        self.prefix_mlp = nn.Sequential(nn.RMSNorm(dim), nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
        self.expert_norm = AdaRMSNorm(dim)
        self.expert_qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.expert_out = nn.Linear(dim, dim, bias=False)
        self.expert_mlp_norm = AdaRMSNorm(dim)
        self.expert_mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def qkv(self, x, projection):
        batch, length, dim = x.shape
        qkv = projection(x).view(batch, length, 3, self.heads, self.head_dim)
        return qkv.permute(2, 0, 3, 1, 4).unbind(0)

    def forward(self, prefix, suffix, condition):
        pq, pk, pv = self.qkv(self.prefix_norm(prefix), self.prefix_qkv)
        eq, ek, ev = self.qkv(self.expert_norm(suffix, condition), self.expert_qkv)
        q = torch.cat([pq, eq], dim=2)
        k = torch.cat([pk, ek], dim=2)
        v = torch.cat([pv, ev], dim=2)
        mask = pi0_mask(prefix.size(1), suffix.size(1), prefix.device)
        attended = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        p_len = prefix.size(1)
        p = attended[:, :, :p_len].transpose(1, 2).contiguous().flatten(2)
        e = attended[:, :, p_len:].transpose(1, 2).contiguous().flatten(2)
        prefix = prefix + self.prefix_out(p)
        suffix = suffix + self.expert_out(e)
        prefix = prefix + self.prefix_mlp(prefix)
        suffix = suffix + self.expert_mlp(self.expert_mlp_norm(suffix, condition))
        return prefix, suffix, mask


class TinyPi0FlowPolicy(nn.Module):
    def __init__(self, dim=24, horizon=6, action_dim=3, depth=3):
        super().__init__()
        self.horizon = horizon
        self.action_dim = action_dim
        self.vision = nn.Linear(10, dim)
        self.language = nn.Embedding(32, dim)
        self.state = nn.Linear(6, dim)
        self.action = nn.Linear(action_dim, dim)
        self.time = nn.Sequential(nn.Linear(1, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.blocks = nn.ModuleList([Pi0JointBlock(dim, 3) for _ in range(depth)])
        self.velocity = nn.Linear(dim, action_dim)

    def forward(self, vision, language, state, noisy_actions, t):
        prefix = torch.cat([self.vision(vision), self.language(language)], dim=1)
        condition = self.time(t[:, None])
        suffix = torch.cat([self.state(state).unsqueeze(1), condition.unsqueeze(1), self.action(noisy_actions)], dim=1)
        mask = None
        for block in self.blocks:
            prefix, suffix, mask = block(prefix, suffix, condition)
        return self.velocity(suffix[:, -self.horizon:]), mask


@torch.no_grad()
def sample_pi0(model, vision, language, state, steps=6):
    actions = torch.randn(state.size(0), model.horizon, model.action_dim, device=state.device)
    for step in range(steps, 0, -1):
        t = torch.full((state.size(0),), step / steps, device=state.device)
        velocity, _ = model(vision, language, state, actions, t)
        actions = actions - velocity / steps
    return actions


pi0 = TinyPi0FlowPolicy().to(device)
vision = torch.randn(3, 3, 10, device=device)
language = torch.randint(0, 32, (3, 4), device=device)
state = torch.randn(3, 6, device=device)
actions = torch.randn(3, 6, 3, device=device)
noise = torch.randn_like(actions)
t = torch.rand(3, device=device)
x_t = (1 - t[:, None, None]) * actions + t[:, None, None] * noise
velocity, mask = pi0(vision, language, state, x_t, t)
pi0_loss = F.mse_loss(velocity, noise - actions)
pi0_loss.backward()
print("π0 rollout:", sample_pi0(pi0, vision[:1], language[:1], state[:1]).shape)


## 4. FAST action tokenizer — quantile normalization, DCT, quantization, BPE-style merges, inverse reconstruction


In [ ]:
def dct_matrix(length, device):
    n = torch.arange(length, device=device, dtype=torch.float32)
    k = torch.arange(length, device=device, dtype=torch.float32)[:, None]
    matrix = torch.cos(math.pi / length * (n + 0.5) * k)
    matrix[0] *= math.sqrt(1 / length)
    matrix[1:] *= math.sqrt(2 / length)
    return matrix

def fit_quantile_bounds(action_dataset):
    flattened = action_dataset.reshape(-1, action_dataset.size(-1))
    low = torch.quantile(flattened, 0.01, dim=0)
    high = torch.quantile(flattened, 0.99, dim=0)
    return low, high

def normalize_actions(actions, low, high):
    return 2 * (actions - low) / (high - low).clamp_min(1e-6) - 1

def denormalize_actions(actions, low, high):
    return 0.5 * (actions + 1) * (high - low) + low

def quantize_coefficients(coefficients, scale=64.0):
    return torch.round(coefficients * scale).to(torch.int64)

def dequantize_coefficients(tokens, scale=64.0):
    return tokens.float() / scale

def pair_counts(sequence):
    return Counter(zip(sequence[:-1], sequence[1:]))

def bpe_merge(sequence, pair, new_token):
    output = []
    index = 0
    while index < len(sequence):
        if index + 1 < len(sequence) and (sequence[index], sequence[index+1]) == pair:
            output.append(new_token)
            index += 2
        else:
            output.append(sequence[index])
            index += 1
    return output

action_dataset = torch.randn(32, 8, 3, device=device)
low, high = fit_quantile_bounds(action_dataset)
action_chunk = action_dataset[:1]
normalized = normalize_actions(action_chunk, low, high)
D = dct_matrix(action_chunk.size(1), device)
coefficients = torch.einsum("kt,btd->bkd", D, normalized)
quantized = quantize_coefficients(coefficients)

sequence = quantized.flatten().tolist()
counts = pair_counts(sequence)
if counts:
    most_common_pair = counts.most_common(1)[0][0]
    compressed_sequence = bpe_merge(sequence, most_common_pair, max(sequence) + 1)
else:
    compressed_sequence = sequence

reconstructed_coeff = dequantize_coefficients(quantized)
reconstructed_norm = torch.einsum("kt,bkd->btd", D, reconstructed_coeff)
reconstructed = denormalize_actions(reconstructed_norm, low, high)
print("FAST scalar tokens:", len(sequence), "after one BPE merge:", len(compressed_sequence))
print("roundtrip MSE:", F.mse_loss(reconstructed, action_chunk).item())


## References and provenance

- ACT: CVAE latent + Transformer action-query chunk decoder.
- Diffusion Policy: observation-conditioned temporal U-Net, iterative denoising and receding-horizon execution.
- π0-style: separate VLM/action-expert parameters, asymmetric joint attention and flow-matching action generation.
- FAST: quantile normalization -> DCT -> discretization -> token compression, with inverse reconstruction checked here.
